# Блокнот 13. Свёрточные сети: от свёртки до переноса обучения

Курс «Машинное обучение с нуля», модуль 13. Разделы: 1 — урок 13.1, 2 — урок 13.2, 3 — уроки 13.4 и 13.6, 4 — урок 13.5, 5 — урок 13.3, 6 — урок 13.7. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

В Colab включите видеокарту: «Среда выполнения → Сменить среду выполнения → T4 GPU». Без неё всё работает, но разделы 5–6 идут в уменьшенном виде.

In [ ]:
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.models as tm
from torch import nn
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("COURSE_DATA", "data")  # папка для скачанных наборов
print("устройство:", device, "| OpenCV", cv2.__version__)

## 1. Картинка, каналы и OpenCV (урок 13.1)

Нарисуем свою картинку: синее небо сверху, красная полоса внизу, — сохраним через OpenCV и прочитаем обратно.

In [ ]:
img_rgb = np.zeros((60, 80, 3), np.uint8)
img_rgb[:40] = (60, 120, 230)      # небо: RGB, синий преобладает
img_rgb[40:] = (220, 40, 30)       # полоса: красная
cv2.imwrite("scene.png", img_rgb)  # OpenCV считает, что ему дали BGR!
back = cv2.imread("scene.png")
print("форма:", back.shape, back.dtype, "| пиксель неба после чтения:", back[0, 0])

fig, axes = plt.subplots(1, 2, figsize=(6, 2))
axes[0].imshow(img_rgb)
axes[0].set_title("исходная RGB")
axes[1].imshow(cv2.cvtColor(back, cv2.COLOR_BGR2RGB))
axes[1].set_title("прочитана и переведена в RGB")
for ax in axes:
    ax.axis("off")
plt.show()

**Задание 1.** Переставьте каналы BGR → RGB срезом NumPy (без `cvtColor`) и проверьте, что получилось то же самое. Затем уменьшите картинку до 40 в ширину и 30 в высоту через `cv2.resize`.

In [ ]:
manual = ...  # ваш код: срез по последней оси
small = ...   # ваш код: cv2.resize, размер (ширина, высота)
assert np.array_equal(manual, cv2.cvtColor(back, cv2.COLOR_BGR2RGB))
assert small.shape == (30, 40, 3)
print("каналы и размер в порядке:", small.shape)

## 2. Свёртка своими руками (урок 13.2)

**Задание 2.** Допишите свёртку одного канала с шагом и отступом. Проверка — сравнение с `F.conv2d` из PyTorch.

In [ ]:
def conv2d(x, k, stride=1, padding=0):
    x = np.pad(x, padding)
    K = k.shape[0]
    out_h = (x.shape[0] - K) // stride + 1
    out_w = (x.shape[1] - K) // stride + 1
    y = np.zeros((out_h, out_w))
    ...  # ваш код: двойной цикл по позициям
    return y


rng = np.random.default_rng(0)
x = rng.normal(size=(9, 11))
k = rng.normal(size=(3, 3))
for stride, padding in ((1, 0), (2, 1), (3, 2)):
    mine = conv2d(x, k, stride, padding)
    ref = F.conv2d(torch.tensor(x)[None, None], torch.tensor(k)[None, None], stride=stride, padding=padding)[0, 0].numpy()
    assert np.allclose(mine, ref), (stride, padding)
    print(f"шаг {stride}, отступ {padding}: выход {mine.shape} совпадает с F.conv2d")

Фильтры Собеля на настоящей картинке из Fashion-MNIST: так выглядят признаки «вертикальный край» и «горизонтальный край».

In [ ]:
fashion_test = datasets.FashionMNIST(DATA, train=False, download=True)
boot = fashion_test.data[0].numpy() / 255
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, (title, im) in zip(axes, (("вход", boot), ("Собель по x", conv2d(boot, sobel_x, padding=1)), ("Собель по y", conv2d(boot, sobel_x.T, padding=1)))):
    ax.imshow(im, cmap="gray_r" if title == "вход" else "RdBu_r")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 3. Архитектуры из torchvision (уроки 13.4 и 13.6)

Создадим сети без весов (ничего не скачивается) и посчитаем параметры и время прохода одной картинки.

In [ ]:
def count(model):
    return sum(p.numel() for p in model.parameters())


x224 = torch.randn(1, 3, 224, 224)
for name in ("alexnet", "vgg16", "googlenet", "resnet18", "resnet50", "densenet121", "mobilenet_v2", "efficientnet_b0", "convnext_tiny"):
    m = tm.get_model(name, weights=None, **({"aux_logits": False, "init_weights": False} if name == "googlenet" else {})).eval()
    with torch.no_grad():
        t0 = time.time()
        m(x224)
        ms = (time.time() - t0) * 1000
    print(f"{name:16s} {count(m) / 1e6:6.1f} млн параметров, {ms:6.0f} мс на картинку (CPU/GPU этой машины)")

Как меняется форма тензора по слоям ResNet-18: печатаем выход каждой части.

In [ ]:
r18 = tm.resnet18(weights=None).eval()
h = x224
with torch.no_grad():
    for name, layer in r18.named_children():
        if name == "fc":
            h = torch.flatten(h, 1)
        h = layer(h)
        print(f"{name:8s} → {tuple(h.shape)}")

## 4. ResNet-18 послойно (урок 13.5)

**Задание 3.** Допишите `forward` базового блока: две свёртки с BatchNorm, ReLU после первой, затем сложение с обходом `self.skip(x)` и ReLU.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.skip = nn.Identity()
        if stride != 1 or c_in != c_out:
            self.skip = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        ...  # ваш код


def resnet18(num_classes=1000):
    layers = [nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, 2, 1)]
    c = 64
    for c_out, stride in [(64, 1), (128, 2), (256, 2), (512, 2)]:
        layers += [BasicBlock(c, c_out, stride), BasicBlock(c_out, c_out)]
        c = c_out
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, num_classes)]
    return nn.Sequential(*layers)


mine = resnet18().eval()
print("наша ResNet-18:", count(mine), "| torchvision:", count(r18))
assert count(mine) == count(r18) == 11_689_512

Самая строгая проверка: перенесём веса torchvision в нашу сеть (порядок слоёв совпадает) и сравним выходы на одной и той же картинке.

In [ ]:
src, dst = r18.state_dict(), mine.state_dict()
assert [v.shape for v in src.values()] == [v.shape for v in dst.values()]
mine.load_state_dict(dict(zip(dst.keys(), src.values())))
with torch.no_grad():
    diff = (mine(x224) - r18(x224)).abs().max().item()
print("максимальная разница выходов:", diff)
assert diff < 1e-4

## 5. Свёрточная сеть против MLP на Fashion-MNIST (урок 13.3)

Две свёртки 3 × 3 с пулингом и два полносвязных слоя. В модуле 12 MLP дал 0,890 за 10 эпох.

**Задание 4.** Посчитайте, сколько чисел приходит на вход `nn.Linear` после двух пулингов (каналы × высота × ширина).

In [ ]:
flat = ...  # ваш код: 64 канала, картинка 28 → 14 → 7
cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(flat, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
print("параметров:", count(cnn))
assert cnn(torch.zeros(2, 1, 28, 28, device=device)).shape == (2, 10)

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
train = datasets.FashionMNIST(DATA, train=True, download=True, transform=tf)
test = datasets.FashionMNIST(DATA, train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=1000)


def evaluate(model, loader):
    model.eval()
    correct = n = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            n += len(y)
    return correct / n


def train_epochs(model, loader, eval_loader, epochs, lr=1e-3, params=None):
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(model(X), y).backward()
            opt.step()
        print(f"эпоха {epoch + 1}: тест {evaluate(model, eval_loader):.4f} ({time.time() - t0:.0f} с)")


EPOCHS = 10 if device == "cuda" else 2
train_epochs(cnn, train_loader, test_loader, EPOCHS)
acc = evaluate(cnn, test_loader)
assert acc > 0.88
print("CNN:", acc)

Что «видят» слои: карты признаков первого и второго свёрточного слоя для одной картинки.

In [ ]:
x0 = test[0][0][None].to(device)
with torch.no_grad():
    a1 = cnn[1](cnn[0](x0))[0].cpu()
    a2 = cnn[4](cnn[3](cnn[2](cnn[1](cnn[0](x0)))))[0].cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for i in range(8):
    axes[0, i].imshow(a1[i], cmap="magma")
    axes[1, i].imshow(a2[i], cmap="magma")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
axes[0, 0].set_title("слой 1", loc="left")
axes[1, 0].set_title("слой 2", loc="left")
plt.show()

## 6. Перенос обучения: ResNet-18 с ImageNet на CIFAR-10 (урок 13.7)

CIFAR-10 — 60 000 цветных картинок 32 × 32, 10 классов (самолёт, автомобиль, птица, кошка, …). Берём небольшую часть: так же бывает в жизни, когда своих данных мало. Скачивание набора (170 МБ) и весов ResNet-18 (45 МБ) занимает пару минут.

In [ ]:
N_TRAIN, N_TEST = (5000, 2000) if device == "cuda" else (1000, 500)
weights = tm.ResNet18_Weights.IMAGENET1K_V1
norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # статистики ImageNet
tf224 = transforms.Compose([transforms.Resize(224), transforms.ToTensor(), norm])
cifar_train = datasets.CIFAR10(DATA, train=True, download=True, transform=tf224)
cifar_test = datasets.CIFAR10(DATA, train=False, download=True, transform=tf224)
g = torch.Generator().manual_seed(0)
sub_train = torch.utils.data.Subset(cifar_train, torch.randperm(len(cifar_train), generator=g)[:N_TRAIN].tolist())
sub_test = torch.utils.data.Subset(cifar_test, torch.randperm(len(cifar_test), generator=g)[:N_TEST].tolist())
print(cifar_train.classes)

**Задание 5.** Заморозьте все веса предобученной ResNet-18 (`requires_grad = False`) и замените последний слой `fc` новым `nn.Linear(512, 10)` — учиться будет только он.

In [ ]:
model = tm.resnet18(weights=weights)
...  # ваш код: заморозка и новая голова
model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("обучаемых параметров:", trainable)
assert trainable == 5130

In [ ]:
loader_tr = torch.utils.data.DataLoader(sub_train, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(sub_test, batch_size=100)
train_epochs(model, loader_tr, val_loader, 3, lr=2e-3, params=model.fc.parameters())
frozen_acc = evaluate(model, val_loader)
print("заморожена, учим голову:", frozen_acc)
assert frozen_acc > 0.6

На видеокарте можно сделать следующий шаг — **дообучение**: разморозить последнюю стадию (`layer4`) и учить её вместе с головой с маленьким шагом. На процессоре это долго, поэтому ячейка выполняется только с GPU.

In [ ]:
if device == "cuda":
    for p in model.layer4.parameters():
        p.requires_grad = True
    params = [{"params": model.layer4.parameters(), "lr": 1e-4}, {"params": model.fc.parameters(), "lr": 1e-3}]
    train_epochs(model, loader_tr, val_loader, 3, lr=1e-4, params=params)
    print("дообучение layer4 + голова:", evaluate(model, val_loader))
else:
    print("Дообучение пропущено: нужна видеокарта.")